# Assignment 05 · Notebook 03: CIFAR-10, một CNN cơ bản và ba mô hình phát triển

**Sinh viên:** Nguyễn Duy Nghĩa · B23DCCN600 · D23CTPM01 · **GVHD:** PGS.TS Trần Đình Quế

Notebook phục vụ **mục 4** của đề (code một CNN cơ bản và ba mô hình phát triển) trên tập
CIFAR-10. Bốn mô hình được huấn luyện theo đúng một cấu hình, mỗi bước chỉ thêm một cơ chế:

| | Mô hình | Cơ chế thêm vào |
|---|---|---|
| M0 | BasicCNN | gốc: [Conv → ReLU → MaxPool] × 3 → FC → FC |
| M1 | VGGNet | chồng conv 3×3, BatchNorm, GAP |
| M2 | ResNet | đường tắt $y = \mathrm{ReLU}(F(x) + x)$ |
| M3 | SE-ResNet | chú ý theo kênh $x \odot \sigma(\mathrm{MLP}(\mathrm{GAP}(x)))$ |

Kết quả ghi ra `outputs/metrics/cifar10.json`, đường học ra `outputs/figdata/cifar10_*_history.dat`,
checkpoint ra `models/cifar10_*.pt`.

In [1]:
import sys
sys.path.insert(0, "..")
import pandas as pd
import torch
from a05.data import GPUImageData
from a05.experiment import run_image_experiment
from a05.models import MODEL_LABELS
from a05.train import IMAGE_CFG

torch.backends.cudnn.benchmark = True
device = torch.device("cuda")
print("Thiết bị:", torch.cuda.get_device_name(device), "· PyTorch", torch.__version__)
print("Cấu hình:", IMAGE_CFG)

Thiết bị: NVIDIA GeForce RTX 4060 Laptop GPU · PyTorch 2.13.0+cu126
Cấu hình: {'epochs': 30, 'batch_size': 256, 'max_lr': 0.1, 'momentum': 0.9, 'weight_decay': 0.0005, 'grad_clip': 2.0}


## Nạp dữ liệu lên GPU

50 000 ảnh huấn luyện được tách 45 000 train / 5 000 validation (phân tầng, seed 42); 10 000 ảnh
test chính thức giữ nguyên. Cả ba nhánh nằm trên VRAM dưới dạng `uint8`.

In [2]:
data = GPUImageData("cifar10", device)
print({k: tuple(v.shape) for k, v in data.x.items()})
print("mean theo kênh:", data.mean.flatten().cpu().numpy().round(4), "std:", data.std.flatten().cpu().numpy().round(4))

{'train': (45000, 3, 32, 32), 'val': (5000, 3, 32, 32), 'test': (10000, 3, 32, 32)}
mean theo kênh: [0.4911 0.4821 0.4464] std: [0.2469 0.2434 0.2616]


## Huấn luyện bốn mô hình

Mỗi mô hình: SGD nesterov (momentum 0,9, weight decay 5e-4), lịch OneCycle với lr cực đại 0,1,
30 epoch, lô 256, AMP fp16. Checkpoint giữ epoch có val loss nhỏ nhất.

In [3]:
results = run_image_experiment("cifar10", data)

== cifar10 / basic


  epoch  1  train 1.7657/0.3598  val 1.4256/0.4846  10.4s


  epoch  2  train 1.4238/0.4909  val 1.2195/0.5616  1.0s


  epoch  3  train 1.2758/0.5509  val 1.0595/0.6316  1.0s


  epoch  4  train 1.1669/0.5893  val 1.0648/0.6326  1.0s


...

  epoch 27  train 0.0475/0.9867  val 0.2534/0.9262  60.7s


  epoch 28  train 0.0341/0.9913  val 0.2473/0.9304  60.6s


  epoch 29  train 0.0292/0.9939  val 0.2524/0.9300  60.2s


  epoch 30  train 0.0262/0.9948  val 0.2489/0.9310  60.4s


   test acc 0.9205  top5 0.9983  macro-F1 0.9203


## Tổng hợp

In [4]:
df = pd.DataFrame(results).T
df.index = [MODEL_LABELS[n] for n in df.index]
df[["acc", "macro_f1", "top5"]] *= 100
df[["acc", "macro_f1", "params", "macs", "epoch_s", "infer_ms", "best_epoch"]].round(3)

,acc,macro_f1,params,macs,epoch_s,infer_ms,best_epoch
BasicCNN,85.62,85.604,620362.0,10848768.0,1.998,0.005,30.0
VGGNet,92.65,92.649,2698954.0,379259392.0,50.077,0.099,30.0
ResNet,92.69,92.685,2741002.0,383650304.0,62.396,0.122,30.0
SE-ResNet,92.05,92.032,2763458.0,383671808.0,63.427,0.115,28.0
